In [1]:
import pandas as pd
import numpy as np

from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load city-level data with lat/lon
city_path = "data/city_safety/city_safety_2023_geo.csv"
city_geo = pd.read_csv(city_path)

city_geo.head()
city_geo.columns

Index(['Rank', 'city', 'country', 'crime_index', 'safety_index', 'year',
       'city_norm', 'country_norm', 'lat', 'lon'],
      dtype='object')

In [3]:
# Keep rows with the core columns present
core_cols = ["city", "country", "lat", "lon", "crime_index", "safety_index"]
city_geo = city_geo.dropna(subset=["lat", "lon", "crime_index", "safety_index"]).copy()

# Features: lat/lon only for baseline
X = city_geo[["lat", "lon"]].values

# Targets: crime and safety indices
y_crime = city_geo["crime_index"].values
y_safety = city_geo["safety_index"].values

X.shape, y_crime.shape

((416, 2), (416,))

In [4]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_crime,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

# Simple geo-KNN: use distance-weighted neighbors in lat/lon space
knn_crime = KNeighborsRegressor(
    n_neighbors=10,        # can tune later
    weights="distance",   # closer cities count more
    metric="haversine"    # great circle distance
)

In [5]:
def to_radians(X_deg: np.ndarray) -> np.ndarray:
    """Convert [lat, lon] in degrees to radians for haversine metric."""
    return np.radians(X_deg)

# Fit
knn_crime.fit(to_radians(X_train), y_train)

# Predict on test
y_pred = knn_crime.predict(to_radians(X_test))

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"KNN crime_index baseline — RMSE: {rmse:.2f}, R^2: {r2:.3f}")

KNN crime_index baseline — RMSE: 12.36, R^2: 0.517


In [6]:
# Small wrapper so cross_val_score can use haversine + radian conversion
from sklearn.base import BaseEstimator, RegressorMixin, clone

class RadianKNNRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_knn=None):
        if base_knn is None:
            base_knn = KNeighborsRegressor(
                n_neighbors=10,
                weights="distance",
                metric="haversine",
            )
        self.base_knn = base_knn

    def fit(self, X, y):
        self.model_ = clone(self.base_knn)
        self.model_.fit(to_radians(X), y)
        return self

    def predict(self, X):
        return self.model_.predict(to_radians(X))

# 5-fold CV on crime_index
rk_crime = RadianKNNRegressor()
cv_rmse = -cross_val_score(
    rk_crime,
    X,
    y_crime,
    scoring="neg_root_mean_squared_error",
    cv=5,
)

print(f"5-fold CV RMSE (crime_index): {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}")

5-fold CV RMSE (crime_index): 15.68 ± 5.22


In [7]:
# Repeat for y saftey
cv_rmse_safety = -cross_val_score(
    rk_crime,
    X,
    y_safety,
    scoring="neg_root_mean_squared_error",
    cv=5,
)

print(f"5-fold CV RMSE (safety_index): {cv_rmse_safety.mean():.2f} ± {cv_rmse_safety.std():.2f}")

5-fold CV RMSE (safety_index): 15.68 ± 5.22


In [8]:
# Fit final models on all data
rk_crime_full = RadianKNNRegressor().fit(X, y_crime)
rk_safety_full = RadianKNNRegressor().fit(X, y_safety)

def estimate_regional_safety(lat, lon, k=10, max_radius_km=None):
    """
    Estimate regional crime and safety indices around an arbitrary lat/lon.

    Returns:
      dict with crime_index, safety_index, and some neighbor diagnostics.
    """
    point = np.array([[lat, lon]])
    # Use the underlying KNN to access neighbors
    knn = rk_crime_full.model_
    distances, indices = knn.kneighbors(
        to_radians(point),
        n_neighbors=k,
        return_distance=True,
    )
    distances = distances[0]  # shape (k,)
    indices = indices[0]

    # Convert haversine distance (radians on unit sphere) to km: d * R_earth
    R_earth_km = 6371.0
    d_km = distances * R_earth_km

    if max_radius_km is not None:
        mask = d_km <= max_radius_km
        if not mask.any():
            return {
                "crime_index": None,
                "safety_index": None,
                "num_neighbors": 0,
                "max_radius_km": max_radius_km,
            }
        d_km = d_km[mask]
        indices = indices[mask]

    # Distance-weighted average (add tiny epsilon to avoid divide-by-zero)
    eps = 1e-6
    weights = 1.0 / (d_km + eps)

    crime_vals = y_crime[indices]
    safety_vals = y_safety[indices]

    crime_est = np.average(crime_vals, weights=weights)
    safety_est = np.average(safety_vals, weights=weights)

    neighbors = city_geo.iloc[indices][["city", "country", "crime_index", "safety_index"]].copy()
    neighbors["distance_km"] = d_km

    return {
        "crime_index": float(crime_est),
        "safety_index": float(safety_est),
        "num_neighbors": int(len(indices)),
        "neighbors": neighbors.reset_index(drop=True),
    }

In [9]:
# Example: some point near Cape Town, South Africa
result = estimate_regional_safety(lat=-33.92, lon=18.42, k=10, max_radius_km=250)
result["crime_index"], result["safety_index"]
result["neighbors"].head()

,city,country,crime_index,safety_index,distance_km
0,Cape Town,South Africa,73.6,26.4,1.014821


In [10]:
# Need indices of the test rows relative to city_geo
_, X_test_idx, _, y_test_idx = train_test_split(
    np.arange(len(X)),
    np.arange(len(X)),
    test_size=0.2,
    random_state=42,
)

# Predict on the same test set
y_pred = knn_crime.predict(to_radians(X_test))

# Pick 10 random test indices
rng = np.random.default_rng(42)
sample_ids = rng.choice(len(X_test_idx), size=10, replace=False)

rows = []
for i in sample_ids:
    global_idx = X_test_idx[i]
    row = city_geo.iloc[global_idx]
    rows.append({
        "city": row["city"],
        "country": row["country"],
        "true_crime_index": y_test[i],
        "pred_crime_index": y_pred[i],
        "abs_error": abs(y_test[i] - y_pred[i]),
    })

eval_df = pd.DataFrame(rows).sort_values("abs_error", ascending=False)
eval_df

,city,country,true_crime_index,pred_crime_index,abs_error
5,Amadora,Portugal,54.6,30.587116,24.012884
6,Kingston,Jamaica,70.9,53.338183,17.561817
3,Queretaro (Santiago de Querétaro),Mexico,37.6,54.021424,16.421424
4,Bern,Switzerland,17.7,33.468551,15.768551
0,Memphis,TN,74.8,59.694555,15.105445
2,Lausanne,Switzerland,26.2,36.854765,10.654765
8,Manchester,United Kingdom,55.4,47.872181,7.527819
1,Krakow (Cracow),Poland,25.8,30.921098,5.121098
7,Christchurch,New Zealand,44.3,40.059087,4.240913
9,Dusseldorf,Germany,35.4,37.729343,2.329343


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, R^2: {r2:.3f}")

MAE: 10.13, RMSE: 12.36, R^2: 0.517


MAE ≈ 10.1
On average, predictions are off by about 10 crime‑index points.
If the index runs roughly 0–100, that’s a “one‑bucket” kind of error (e.g., predicting 45 when true is 55).

RMSE ≈ 12.4
RMSE penalizes big errors more. Being larger than MAE means you do have some cities where the error is 20+ points (exactly what you saw in the table). Those outliers dominate RMSE and are the cases where pure lat/lon fails (e.g., Kingston vs Bern).

R² ≈ 0.52
The model explains about 52% of the variance in city crime scores compared with predicting a constant mean for all cities. That’s decent for a super‑simple model using only coordinates, but it also means almost half of the variability is still unexplained.

## Testing different ks and weights before adding new features and other tweaks 

In [13]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def eval_knn_haversine(k, weights):
    model = KNeighborsRegressor(
        n_neighbors=k,
        weights=weights,
        metric="haversine",
    )
    model.fit(to_radians(X_train), y_train)
    y_pred = model.predict(to_radians(X_test))
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    return mae, rmse, r2

ks = [3, 5, 10, 20]
weight_options = ["uniform", "distance"]

results = []
for k in ks:
    for w in weight_options:
        mae, rmse, r2 = eval_knn_haversine(k, w)
        results.append({
            "k": k,
            "weights": w,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

import pandas as pd
results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


,k,weights,MAE,RMSE,R2
3,5,distance,9.808432,12.103851,0.536718
5,10,distance,10.134324,12.355431,0.517259
1,3,distance,9.992510,12.378956,0.515419
2,5,uniform,10.180000,12.391880,0.514407
0,3,uniform,10.082143,12.459112,0.509123
7,20,distance,10.317301,12.745162,0.486325
4,10,uniform,11.048929,13.381142,0.433781
6,20,uniform,11.987560,14.138321,0.367888


In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path

# PDF 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_KNN.ipynb to pdf
[NbConvertApp] Writing 50221 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 57329 bytes to Geo_KNN.pdf
